<a href="https://colab.research.google.com/github/yasumorishima/mlb-data-analysis/blob/main/notebooks/sql/mlb_home_run_race_2024_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLB Home Run Race 2024 (SQL Version)

Animated visualization of the 2024 MLB home run race using **DuckDB SQL** for data aggregation.

(2024年MLBホームラン競争の**SQL版**アニメーション可視化)

## SQL Skills Demonstrated:
- Window functions (`SUM() OVER`)
- `PIVOT` operations
- CTEs (Common Table Expressions)
- Date range generation
- Cumulative aggregation

In [ ]:
!pip install pybaseball duckdb bar_chart_race -q
!apt-get update --quiet && apt-get install -y ffmpeg -qq

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pybaseball import statcast, playerid_reverse_lookup
import duckdb
import bar_chart_race as bcr
from IPython.display import Video

## 1. Data Acquisition (データ取得)

In [ ]:
# Fetch 2024 season Statcast data
print("Fetching Statcast data...")
df = statcast(start_dt='2024-03-20', end_dt='2024-10-01')
print(f"Total records: {len(df):,}")

In [4]:
# DuckDB connection
con = duckdb.connect()

## 2. Filter Regular Season Home Runs (レギュラーシーズンHR抽出)

In [5]:
# Extract home runs from regular season using SQL
df_hr = con.execute("""
    SELECT
        game_date,
        batter,
        events
    FROM df
    WHERE game_type = 'R'
      AND events = 'home_run'
    ORDER BY game_date, batter
""").df()

print(f"Total home runs in 2024 regular season: {len(df_hr):,}")

Total home runs in 2024 regular season: 5,453


## 3. Daily HR Count by Batter (日別打者ごとのHR数)

In [6]:
# Aggregate HR count by date and batter using SQL
df_daily_hr = con.execute("""
    SELECT
        game_date,
        batter,
        COUNT(*) as hr_count
    FROM df
    WHERE game_type = 'R'
      AND events = 'home_run'
    GROUP BY game_date, batter
    ORDER BY game_date, batter
""").df()

print("Daily HR counts:")
print(df_daily_hr.head(10))

Daily HR counts:
   game_date  batter  hr_count
0 2024-03-21  592518         1
1 2024-03-21  605141         1
2 2024-03-28  502671         1
3 2024-03-28  518692         1
4 2024-03-28  543807         1
5 2024-03-28  545361         1
6 2024-03-28  571745         1
7 2024-03-28  592696         1
8 2024-03-28  605141         1
9 2024-03-28  605361         2


## 4. Cumulative HR by Batter (打者別累積HR - Window Function)

In [7]:
# Calculate cumulative HR using SQL window functions
df_cumulative = con.execute("""
    WITH daily_hr AS (
        SELECT
            game_date,
            batter,
            COUNT(*) as hr_count
        FROM df
        WHERE game_type = 'R'
          AND events = 'home_run'
        GROUP BY game_date, batter
    )
    SELECT
        game_date,
        batter,
        hr_count,
        SUM(hr_count) OVER (
            PARTITION BY batter
            ORDER BY game_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) as cumulative_hr
    FROM daily_hr
    ORDER BY game_date, batter
""").df()

print("Cumulative HR (sample):")
print(df_cumulative.head(20))

Cumulative HR (sample):
    game_date  batter  hr_count  cumulative_hr
0  2024-03-21  592518         1            1.0
1  2024-03-21  605141         1            1.0
2  2024-03-28  502671         1            1.0
3  2024-03-28  518692         1            1.0
4  2024-03-28  543807         1            1.0
5  2024-03-28  545361         1            1.0
6  2024-03-28  571745         1            1.0
7  2024-03-28  592696         1            1.0
8  2024-03-28  605141         1            2.0
9  2024-03-28  605361         2            2.0
10 2024-03-28  608671         1            1.0
11 2024-03-28  623993         1            1.0
12 2024-03-28  624415         1            1.0
13 2024-03-28  624424         1            1.0
14 2024-03-28  641933         1            1.0
15 2024-03-28  646240         1            1.0
16 2024-03-28  650490         1            1.0
17 2024-03-28  656775         1            1.0
18 2024-03-28  658668         1            1.0
19 2024-03-28  664238         1     

## 5. Final HR Leaders (最終HR順位)

In [8]:
# Get final HR totals using SQL
df_final = con.execute("""
    SELECT
        batter,
        COUNT(*) as total_hr
    FROM df
    WHERE game_type = 'R'
      AND events = 'home_run'
    GROUP BY batter
    ORDER BY total_hr DESC
    LIMIT 15
""").df()

# Map batter IDs to names
name_mapping = {}
for batter_id in df_final['batter']:
    try:
        info = playerid_reverse_lookup([batter_id], key_type='mlbam')
        if not info.empty:
            name_mapping[batter_id] = f"{info['name_first'].values[0]} {info['name_last'].values[0]}"
        else:
            name_mapping[batter_id] = str(batter_id)
    except:
        name_mapping[batter_id] = str(batter_id)

df_final['player_name'] = df_final['batter'].map(name_mapping)

print("2024 HR Leaders:")
print(df_final[['player_name', 'total_hr']].to_string(index=False))

Gathering player lookup table. This may take a moment.
2024 HR Leaders:
      player_name  total_hr
      aaron judge        58
    shohei ohtani        54
anthony santander        44
        juan soto        41
    marcell ozuna        39
     josé ramírez        39
     brent rooker        39
   kyle schwarber        38
 gunnar henderson        37
      ketel marte        36
   yordan álvarez        35
      pete alonso        34
      cal raleigh        34
 francisco lindor        33
teoscar hernández        33


## 6. Pivot Table for Animation (アニメーション用ピボットテーブル)

In [9]:
# Create pivot table using SQL and pandas
# First get all unique dates and batters
df_pivot_data = con.execute("""
    WITH daily_hr AS (
        SELECT
            CAST(game_date AS DATE) as game_date,
            batter,
            COUNT(*) as hr_count
        FROM df
        WHERE game_type = 'R'
          AND events = 'home_run'
        GROUP BY game_date, batter
    )
    SELECT * FROM daily_hr
""").df()

# Pivot and cumsum in pandas (DuckDB's PIVOT is limited)
df_pivot = df_pivot_data.pivot(index='game_date', columns='batter', values='hr_count').fillna(0)
df_pivot.index = pd.to_datetime(df_pivot.index)
df_pivot.sort_index(inplace=True)

# Reindex to fill missing dates
all_dates = pd.date_range(start='2024-03-20', end='2024-10-01')
df_pivot = df_pivot.reindex(all_dates, fill_value=0)

# Cumulative sum
df_cumsum = df_pivot.cumsum()

# Rename columns to player names
df_cumsum.rename(columns=name_mapping, inplace=True)

print("Pivot table shape:", df_cumsum.shape)
print(df_cumsum.tail())

Pivot table shape: (196, 522)
batter      444482  453568  455117  456781  457705  457759  467793  500743  \
2024-09-27     8.0    12.0     4.0     7.0    20.0    11.0    22.0     6.0   
2024-09-28     8.0    12.0     4.0     8.0    20.0    11.0    22.0     6.0   
2024-09-29     8.0    12.0     4.0     8.0    20.0    11.0    23.0     6.0   
2024-09-30     8.0    12.0     4.0     8.0    20.0    11.0    23.0     6.0   
2024-10-01     8.0    12.0     4.0     8.0    20.0    11.0    23.0     6.0   

batter      501303  502054  ...  696285  700242  700250  700932  701305  \
2024-09-27     1.0     9.0  ...     3.0     4.0     7.0     5.0     2.0   
2024-09-28     1.0     9.0  ...     3.0     4.0     7.0     5.0     2.0   
2024-09-29     1.0     9.0  ...     3.0     4.0     7.0     5.0     2.0   
2024-09-30     1.0     9.0  ...     3.0     4.0     7.0     5.0     2.0   
2024-10-01     1.0     9.0  ...     3.0     4.0     7.0     5.0     2.0   

batter      701538  702358  702616  807799  808982

## 7. Create Bar Chart Race Animation (バーチャートレースアニメーション作成)

In [ ]:
# Generate bar chart race animation
output_filename = 'hr_race_2024_sql.mp4'

bcr.bar_chart_race(
    df=df_cumsum,
    filename=output_filename,
    n_bars=10,
    period_fmt='%Y-%m-%d',
    title='2024 MLB Home Run Race (SQL Version)',
    filter_column_colors=True,
)

print(f"Animation saved: {output_filename}")

In [11]:
# Display the video
display(Video(output_filename, embed=True))

## SQL Techniques Used

1. **CTE (WITH clause)**: Organized complex queries into readable steps
2. **Window Functions**: `SUM() OVER (PARTITION BY ... ORDER BY ...)` for cumulative totals
3. **Aggregation**: `GROUP BY` with `COUNT(*)` for daily HR counts
4. **Filtering**: `WHERE` with multiple conditions
5. **Ordering & Limiting**: `ORDER BY ... LIMIT` for leaderboard